In [3]:
import time
import numpy as np
import psi4

def diag_lps(diag, A, nel):
    Fp = psi4.core.triplet(A, diag, A, True, False, True)
    nbf = A.shape[0]
    Cp = psi4.core.Matrix(nbf, nbf)
    eigvals = psi4.core.Vector(nbf)
    Fp.diagonalize(Cp, eigvals, psi4.core.DiagonalizeOrder.Ascending)
    C = psi4.core.doublet(A, Cp, False, False)
    Cocc = psi4.core.Matrix(nbf, 1) 
    Cocc.np[:] = np.sqrt(nel) * C.np[:, :1]
    D = psi4.core.doublet(Cocc, Cocc, False, True) 
    return D, eigvals.np[0]

def Vpot_init(build_superfunctional, wfn, alias, vname, restricted=True):
    sup = build_superfunctional(alias, restricted)[0]
    sup.set_deriv(1)
    sup.allocate()
    Vpot = psi4.core.VBase.build(wfn.basisset(), sup, vname)
    return Vpot

def Vpot_builder(Vpot, D, V, D_half):
    D_half.copy(D)
    D_half.scale(0.5)
    Vpot.set_D([ D_half ])
    Vpot.compute_V([ V ])
    e = Vpot.quadrature_values()['FUNCTIONAL']
    return e, V

def lps_solver(maxiter, Pauli, XC, lam, mol, damp, FA, D_guess=None, DIIS=True):
    
    E_conv = 1.0e-5
    D_conv = 1.0e-5
    
    wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option("BASIS"))
    mints = psi4.core.MintsHelper(wfn.basisset())
    S = mints.ao_overlap()
    
    nbf = wfn.nso()
    nel = wfn.nalpha() + wfn.nbeta()

    print('Number of basis functions:   %d' % nbf)

    build_superfunctional = psi4.driver.dft.build_superfunctional
    D_half = psi4.core.Matrix(nbf, nbf)

    VPpot = Vpot_init(build_superfunctional, wfn, Pauli, "RV", restricted=True)
    VPpot.initialize()
    VP_null = psi4.core.Matrix(nbf, nbf)

    VXCpot = Vpot_init(build_superfunctional, wfn, XC, "RV", restricted=True)
    VXCpot.initialize()
    VXC_null = psi4.core.Matrix(nbf, nbf)

    vW = {
        "name": "vW",
        "x_functionals": {"LDA_X": {"alpha": 0.00}},
        "c_functionals": {"GGA_K_VW": {"alpha": 1.00}}
    }
    VvWpot = Vpot_init(build_superfunctional, wfn, vW, "RV", restricted=True)
    VvWpot.initialize()
    VvW_null = psi4.core.Matrix(nbf, nbf)

    V = mints.ao_potential()
    T = mints.ao_kinetic()
    H = T.clone()
    H.add(V)
    I = np.asarray(mints.ao_eri())
    A = mints.ao_overlap()
    A.power(-0.5, 1.e-14)
    F = psi4.core.Matrix(nbf, nbf)
    J = psi4.core.Matrix(nbf, nbf)
    VG = psi4.core.Matrix(nbf, nbf)
    D_diff = psi4.core.Matrix(nbf, nbf)
    
    if D_guess is not None:
        D = D_guess.clone()
        mu = 0
    else:
        D, mu = diag_lps(H, A, nel)

    Enuc = mol.nuclear_repulsion_energy()
    Eold = 0.0
    
    print('\nStarting SCF iterations:')
    t = time.time()
    conv_list = []

    if DIIS:
        diis_obj = psi4.p4util.solvers.DIIS(max_vec=6, removal_policy="oldest")
   
    print("\n    Iter               Energy         ChemPot       Delta E         dRMS\n")
    for SCF_ITER in range(1, maxiter + 1):
    
        D_old = D
        
        J_np = np.einsum('pqrs,rs->pq', I, D.np, optimize=True)
        J.np[:] = J_np
        F.copy(H)
        F.axpy(1.0, J)
        if FA[0]:
            if nel == 0:
                F.axpy(0.0, J)
            else: 
                F.axpy(-FA[1]/nel, J)

        pau_e, VP = Vpot_builder(VPpot, D, VP_null, D_half)
        xc_e, VXC = Vpot_builder(VXCpot, D, VXC_null, D_half)
        vw_e, VvW = Vpot_builder(VvWpot, D, VvW_null, D_half)
        g_e = pau_e + xc_e + ( lam - 1.0 ) * vw_e 

        VG.copy(VP)
        VG.axpy(1.0, VXC)
        VG.axpy((lam - 1.0), VvW)
        F.axpy(1.0, VG)

        if DIIS:
            diis_e = psi4.core.triplet(F, D, S, False, False, False)
            diis_e.subtract(psi4.core.triplet(S, D, F, False, False, False))
            diis_e = psi4.core.triplet(A, diis_e, A, False, False, False)
        
            diis_obj.add(F, diis_e)
            dRMS = diis_e.rms()

        SCF_E = H.vector_dot(D)
        SCF_E += 0.5 * J.vector_dot(D)
        if FA[0]:
            SCF_E += 0.5 * J.vector_dot(D) * ( - FA[1] / nel )
        SCF_E += g_e
        SCF_E += Enuc

        conv_list.append(np.log10(abs(SCF_E - Eold)))

        if DIIS:
            print('SCF Iter%3d: % 18.8f   % 1.5E   % 1.5E   % 1.5E'
                % (SCF_ITER, SCF_E, mu, (SCF_E - Eold), dRMS))
            
            if (abs(SCF_E - Eold) < E_conv and dRMS < D_conv):
                break
            
            Eold = SCF_E
            F = diis_obj.extrapolate()

        D, mu = diag_lps(F, A, nel)

        if not DIIS:
            D_diff.copy(D)
            D_diff.subtract(D_old)
            dRMS = D_diff.rms()
            print('SCF Iter%3d: % 18.8f   % 1.5E   % 1.5E   % 1.5E'
                % (SCF_ITER, SCF_E, mu, (SCF_E - Eold), dRMS))
            
            if (abs(SCF_E - Eold) < E_conv and dRMS < D_conv):
                break

            Eold = SCF_E
        
        if (dRMS > damp[2]):
            current_damp = damp[0]
        else:
            current_damp = damp[1]
        D.scale(1.0 - current_damp)
        D.axpy(current_damp, D_old)
        
        if SCF_ITER == maxiter:
            SCF_D = D
            print("\nWARNING ! SCF did not converge. The final values are printed")
            return SCF_E, SCF_D, SCF_ITER
    
    SCF_D = D
    
    print('\nTotal time for SCF iterations: %.3f seconds ' % (time.time() - t))

    return SCF_E, SCF_D, SCF_ITER, conv_list

In [5]:
psi4.core.clean_options()
psi4.core.clean()
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'UGBS_S', 
                 'DFT_SPHERICAL_POINTS': 6,
                  'DFT_RADIAL_POINTS': 1000})

Pauli = {
    "name": "Pauli",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_K_TF": {"alpha": 1.00}}
}
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

## Damping factor before cutoff, after cutoff, the cutoff.
damp = [0.9, 0.9, 0.0001]
## Calculate Fermi–Amaldi? Scaling factor.
FA = [False, 1.0]

SCF_E, D, SCF_ITER, conv_list = lps_solver(5000, Pauli, XC, 1.0, mol, damp, FA, D_guess=None, DIIS=True)
print('\nFinal SCF energy: %.4f Hartree' % SCF_E)

Number of basis functions:   31

Starting SCF iterations:

    Iter               Energy         ChemPot       Delta E         dRMS

SCF Iter  1:         1.09953778   -2.00000E+00    1.09954E+00    2.67149E-01
SCF Iter  2:         0.56981114    1.88980E-02   -5.29727E-01    2.21326E-01
SCF Iter  3:         0.15786224   -1.53641E-02   -4.11949E-01    1.83346E-01
SCF Iter  4:        -0.08814430   -4.28182E-01   -2.46007E-01    1.59366E-01
SCF Iter  5:        -0.34242787   -1.98532E-01   -2.54284E-01    1.33459E-01
SCF Iter  6:        -0.55611408   -1.34924E-01   -2.13686E-01    1.10286E-01
SCF Iter  7:        -0.72460852   -9.28653E-02   -1.68494E-01    9.10380E-02
SCF Iter  8:        -0.81210168   -2.07046E-01   -8.74932E-02    8.25473E-02
SCF Iter  9:        -0.89678019   -2.10691E-01   -8.46785E-02    7.34778E-02
SCF Iter 10:        -0.96947309   -2.11499E-01   -7.26929E-02    6.53478E-02
SCF Iter 11:        -1.03295499   -2.07283E-01   -6.34819E-02    5.78968E-02
SCF Iter 12:        